# 02 — Wine Quality EDA Notes

### Learn how to explore a numeric dataset and a quality score

**Level:** beginner → advanced  
**Based on:** the supplied red-wine EDA lecture transcript and notebook.

> **Simple picture:** each row is one bottle of wine. The chemistry columns are clues, and `quality` is the score we want to understand.

## Learning goals

You will learn to inspect the Wine Quality dataset, check its health, study distributions and correlations, compare quality groups, and create careful model-ready features.


## 1. The dataset story

The red Wine Quality dataset contains physicochemical measurements such as acidity, sugar, sulphates, alcohol, and pH. The target column `quality` is an ordered score: a higher score means a better-rated wine.

Most columns are numeric, which makes this a good dataset for practising:

- descriptive statistics;
- histograms and box plots;
- correlation heatmaps;
- comparing features across quality groups;
- checking whether the quality classes are balanced.

### First question

Do not begin by asking “Which model is best?” Begin by asking “What does a bottle with a high quality score look like in this data?”


In [ ]:
# Beginner-friendly guide:
# We load the tools used to read the wine table, calculate summaries, and draw EDA charts.
# The plot style makes every chart easier to read while we learn.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
# Beginner-friendly guide:
# This cell reads the standard UCI red-wine CSV, which usually uses a semicolon instead of a comma.
# Put winequality-red.csv beside this notebook, or change the path below to where you saved the file.
data_path = Path("winequality-red.csv")
if not data_path.exists():
    raise FileNotFoundError("Add winequality-red.csv beside this notebook, then run this cell again.")

wine = pd.read_csv(data_path, sep=";")
print("Rows and columns:", wine.shape)
display(wine.head())


## 2. Start with the data health check

Before making pretty plots, check whether the table is usable.

- `shape` tells how much data you have.
- `info()` reveals data types and non-missing counts.
- `describe()` gives the centre and spread of numeric columns.
- Missing values and duplicate rows may change later conclusions.

### Small but important idea

The **mean** can be pulled by very large values. The **median** is the middle value, so comparing both can reveal skewness or outliers.


In [ ]:
# Beginner-friendly guide:
# We create a compact health report without changing any wine records.
# This lets us see missing values, duplicate bottles, and the usual size of each measurement.
print(wine.info())
print("\nMissing values:\n", wine.isna().sum())
print("\nDuplicate rows:", wine.duplicated().sum())
display(wine.describe().T)


## 3. Univariate analysis: one clue at a time

For each measurement, ask simple questions:

- Is the shape roughly balanced or pulled to one side?
- Are there long tails or unusual values?
- Are there many wines in every quality class, or only a few in some classes?

Histograms show the overall shape. Box plots make outliers easier to notice. Count plots show whether the target classes are imbalanced.


In [ ]:
# Beginner-friendly guide:
# We look at the alcohol distribution and count how many wines received each quality score.
# The second chart is important because a rare quality class can be harder for a model to learn.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(wine["alcohol"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Alcohol distribution")

sns.countplot(data=wine, x="quality", ax=axes[1])
axes[1].set_title("Number of wines at each quality score")

plt.tight_layout()


## 4. Bivariate and multivariate analysis

Now let clues talk to each other. A correlation heatmap is a quick map of linear relationships among numeric columns.

### Read it carefully

- Positive correlation: two values often rise together.
- Negative correlation: one often rises while the other falls.
- Near zero: no strong **linear** pattern was found.

Correlation is not proof of cause. For example, a measurement may correlate with quality because both are connected to another chemical process.


In [ ]:
# Beginner-friendly guide:
# The heatmap compares every numeric pair at once. Darker colours mean a stronger linear relationship.
# The scatter plot then lets us look closely at alcohol and pH while colour shows the quality score.
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(wine.corr(numeric_only=True), cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Correlation heatmap")

sns.scatterplot(data=wine, x="alcohol", y="pH", hue="quality", palette="viridis", ax=axes[1])
axes[1].set_title("Alcohol and pH by quality")

plt.tight_layout()


## 5. From EDA to feature engineering

The quality score is ordered. You can keep it as a multi-class target, or make a simple business question such as “Is this wine good?”

For example, a score of 7 or higher could be called `good_quality = 1`. This changes the problem, so always explain the rule and check class balance again.

### Advanced caution

If you later scale, impute, select features, or handle class imbalance, learn those rules from training data only. The test set should stay unseen until the final evaluation.


In [ ]:
# Beginner-friendly guide:
# We create a clear yes-or-no target from the ordered quality score.
# This is only one possible rule; change the cutoff if your project defines “good” differently.
wine_model = wine.copy()
wine_model["good_quality"] = (wine_model["quality"] >= 7).astype(int)

print("Good-quality class balance:")
display(wine_model["good_quality"].value_counts(normalize=True).rename("proportion"))

# This list keeps the original chemistry columns separate from the new target column.
feature_columns = wine_model.drop(columns=["quality", "good_quality"]).columns.tolist()
print("Model features:", feature_columns)


## End-of-topic recap

1. Confirm that the wine table has the rows and columns you expect.
2. Check missing values and duplicates before trusting plots.
3. Use distributions to understand each chemical measurement.
4. Use quality counts to check target imbalance.
5. Use correlations and scatter plots as clues, not proof.
6. Document every target or feature rule before modelling.

**One-line memory:** EDA tells the story of the wine data; feature engineering prepares that story for a model.
